### Prettraining on unlabeled data

In [1]:
from importlib.metadata import version

pkgs = ["matplotlib",
        "numpy",
        "tiktoken", 
        "torch",
        "tensorflow"    
       ]

for pkg in pkgs:
    print(pkg,version(pkg))


matplotlib 3.10.8
numpy 2.4.4
tiktoken 0.12.0
torch 2.11.0
tensorflow 2.21.0


In [4]:
import torch
from all_input_blocks import GPTModel

GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 256,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers":12,
    "drop_rate": 0.1,
    "qkv_bias": False 
}

In [5]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval();

In [6]:
import tiktoken
from all_input_blocks import generate_text_simple

def text_to_token_ids(text,tokeniser):
    encoded = tokeniser.encode(text,allowed_special = {"|<endoftext>|"})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)
    return encoded_tensor


In [7]:
start_context = "Every effort moves you"
tokeniser = tiktoken.get_encoding("gpt2")
token_ids = text_to_token_ids(start_context,tokeniser)
token_ids

tensor([[6109, 3626, 6100,  345]])

In [8]:
def token_ids_to_text(token_ids,tokeniser):
    flat = token_ids.squeeze(0)
    return tokeniser.decode(flat.tolist())

token_ids_to_text(token_ids,tokeniser)

'Every effort moves you'

In [9]:
token_ids = generate_text_simple(
    model = model,
    idx = text_to_token_ids(start_context,tokeniser),
    max_new_tokens = 10, # no of new words to be generated 
    context_size = GPT_CONFIG_124M["context_length"] 
)

In [10]:
token_ids.shape

torch.Size([1, 14])

In [11]:
token_ids_to_text(token_ids,tokeniser)

'Every effort moves you rentingetic wasnم refres RexMeCHicular stren'

### Calculating text generation loss: cross-entropy and perplexity

In [12]:
inputs = torch.tensor([[16833, 3626, 6100],   # ["every effort moves",
                       [40,    1107, 588]])   #  "I really like"]

targets = torch.tensor([[3626, 6100, 345  ],  # [" effort moves you",
                        [1107,  588, 11311]]) #  " really like chocolate"]

In [13]:
with torch.no_grad():
    logits = model(inputs)

In [14]:
logits.shape

torch.Size([2, 3, 50257])

In [15]:
logits.flatten(0,1).shape


torch.Size([6, 50257])

In [16]:
targets.shape

torch.Size([2, 3])

In [17]:
targets.flatten(0).shape

torch.Size([6])

cross-entropy loss 
1. convert logtis to probability
2. take log of probability
average log of probability
minimise - average log of probability to 0 so that probability tends to 1

In [18]:
logits_flat = logits.flatten(0,1)
targets_flat = targets.flatten(0)

loss = torch.nn.functional.cross_entropy(logits_flat, targets_flat)
loss

tensor(10.7940)

### Calculating the training and validation set losses

In [19]:
with open("the-verdict.txt","r",encoding="utf-8") as f:
    text_data = f.read()

In [20]:
text_data[:99];

In [21]:
total_characters = len(text_data)
total_tokens = len(tokeniser.encode(text_data))
print(f" total_characters: {total_characters}, total_tokens: {total_tokens}")

 total_characters: 20479, total_tokens: 5145


In [22]:
from all_input_blocks import create_dataloader_v1

train_ratio = 0.90
split_idx = int(train_ratio*len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]


In [23]:
torch.manual_seed(123)

train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last = False,
    shuffle=False,
    num_workers=0
)


In [29]:
for x, y in train_loader:
    pass
print(x.shape, y.shape)

torch.Size([2, 256]) torch.Size([2, 256])


512

In [25]:
for x, y in val_loader:
    print(x.shape, y.shape)

torch.Size([2, 256]) torch.Size([2, 256])


In [35]:
#print num of elements in training and validation set
no_train_token = 0
no_val_token = 0
# x input_batch
# y target_batch
for x,y in train_loader:
    no_train_token += x.numel()

for x,y in val_loader:
    no_val_token += x.numel()
    
print(f"No. of Training token: {no_train_token}")
print(f"No. of Validating token: {no_val_token}")
print(f"No. of total token: {no_train_token+no_val_token}")

No. of Training token: 4608
No. of Validating token: 512
No. of total token: 5120


In [49]:
def calc_loss_batch(input_batch,target_batch,model,device):
    input_batch,target_batch = input_batch.to(device),target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0,1),target_batch.flatten(0))
    return loss

def calc_loss_loader(data_loader,model,device,num_batches=None):
    total_loss = 0
    if len(data_loader) ==0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches,len(data_loader))

    for i, (input_batch,target_batch) in enumerate(data_loader):
        if i<num_batches:
            loss = calc_loss_batch(input_batch,target_batch,model,device)
            total_loss += loss.item()
        else:
            break
    return total_loss/num_batches

In [47]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [51]:
torch.manual_seed(123)

with torch.no_grad():
    train_loss = calc_loss_loader(train_loader,model,device)
    val_loss = calc_loss_loader(val_loader,model,device)

print("Train loss:", train_loss)
print("Var loss:", val_loss)

Train loss: 10.98758347829183
Var loss: 10.981106758117676


In [54]:
#Perplexity
torch.exp(torch.tensor(train_loss))

tensor(59135.2930)